In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Configurable settings ──────────────────────────────────────────────
catalog       = "hr_catalog"
schema        = "hr_core"
endpoint_name = "hr-vector-search-endpoint"  # vector search endpoint name
embedding_model = "databricks-gte-large-en"
# ─────────────────────────────────────────────────────────────────────

# 1. Create Vector Search endpoint (skip if it already exists)
try:
    w.vector_search_endpoints.create_endpoint(
        name=endpoint_name,
        endpoint_type="STANDARD"
    )
    print(f"Endpoint '{endpoint_name}' creation started.")
except Exception as e:
    print(f"Endpoint '{endpoint_name}' may already exist: {e}")

# 2. Create two chunk tables (Delta, with CDF + primary key for Delta Sync)
chunk_tables = [
    "hr_document_chunks_200",
    "hr_document_chunks_800",
]

for tbl in chunk_tables:
    full_name = f"{catalog}.{schema}.{tbl}"
    
    # Check if table exists
    try:
        spark.sql(f"DESCRIBE TABLE {full_name}")
        table_exists = True
        print(f"Table '{full_name}' already exists.")
    except:
        table_exists = False
    
    if not table_exists:
        # Create new table with NOT NULL id column
        spark.sql(f"""
            CREATE TABLE {full_name} (
                id        BIGINT NOT NULL,
                content   STRING,
                metadata  STRING,
                chunk_size INT
            )
            USING DELTA
            TBLPROPERTIES (
                delta.enableChangeDataFeed = true
            )
        """)
        # Add primary key constraint
        spark.sql(f"""
            ALTER TABLE {full_name}
            ADD CONSTRAINT pk_{tbl} PRIMARY KEY (id)
        """)
        print(f"Chunk table '{full_name}' created with CDF + PK enabled.")
    else:
        # Table exists - try to alter column to NOT NULL and add constraint if needed
        try:
            spark.sql(f"""
                ALTER TABLE {full_name}
                ALTER COLUMN id SET NOT NULL
            """)
            print(f"Updated '{full_name}' id column to NOT NULL.")
        except Exception as e:
            if "already" not in str(e).lower():
                print(f"Note: {e}")
        
        # Try to add primary key constraint
        try:
            spark.sql(f"""
                ALTER TABLE {full_name}
                ADD CONSTRAINT pk_{tbl} PRIMARY KEY (id)
            """)
            print(f"Added primary key constraint to '{full_name}'.")
        except Exception as e:
            if "already exists" not in str(e).lower() and "already" not in str(e).lower():
                print(f"Note adding constraint: {e}")

# 3. Create two Delta Sync (managed-embeddings) vector search indexes
from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
    VectorIndexType
)

index_specs = [
    ("hr_rag_index_200", chunk_tables[0]),
    ("hr_rag_index_800", chunk_tables[1]),
]

for index_name, source_table in index_specs:
    full_index  = f"{catalog}.{schema}.{index_name}"
    full_source = f"{catalog}.{schema}.{source_table}"
    try:
        # Check if index already exists
        try:
            existing = w.vector_search_indexes.get_index(index_name=full_index)
            print(f"Index '{full_index}' already exists (status: {existing.status.ready}).")
            continue
        except:
            pass  # Index doesn't exist, create it
        
        # Create the index with proper SDK objects (use string for pipeline_type)
        w.vector_search_indexes.create_index(
            name=full_index,
            endpoint_name=endpoint_name,
            primary_key="id",
            index_type=VectorIndexType.DELTA_SYNC,
            delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
                source_table=full_source,
                embedding_source_columns=[
                    EmbeddingSourceColumn(
                        name="content",
                        embedding_model_endpoint_name=embedding_model,
                    )
                ],
                pipeline_type=PipelineType.TRIGGERED
            ),
        )
        print(f"✓ Index '{full_index}' creation started (source: {full_source}).")
    except Exception as e:
        import traceback
        print(f"✗ Error creating index '{full_index}':")
        print(f"   {e}")
        traceback.print_exc()

In [0]:
# Chunk the HR documents into different chunk sizes
import hashlib
import json

# Get the source document chunks (already parsed)
source_chunks = spark.table("hr_catalog.hr_core.hr_document_chunks").collect()

print(f"Source document chunks: {len(source_chunks)}")

def chunk_text(text, chunk_size, overlap=50):
    """Split text into chunks with overlap"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():  # Only add non-empty chunks
            chunks.append(chunk)
        start = end - overlap  # Move back by overlap amount
        if start >= len(text) - overlap:
            break
    return chunks

# Process for chunk size 200
chunks_200 = []
for row in source_chunks:
    doc_name = row['document_name']
    content = row['chunk_text']  # Use chunk_text column
    sub_chunks = chunk_text(content, 200)
    
    for i, chunk in enumerate(sub_chunks):
        # Create unique ID using hash
        unique_str = f"{doc_name}_{row['chunk_id']}_{i}_{chunk[:30]}"
        chunk_id = int(hashlib.md5(unique_str.encode()).hexdigest()[:15], 16)
        
        chunks_200.append({
            'id': chunk_id,
            'content': chunk,
            'metadata': json.dumps({
                "document_name": doc_name,
                "source_chunk_id": int(row['chunk_id']),
                "sub_chunk_index": i
            }),
            'chunk_size': 200
        })

# Process for chunk size 800
chunks_800 = []
for row in source_chunks:
    doc_name = row['document_name']
    content = row['chunk_text']  # Use chunk_text column
    sub_chunks = chunk_text(content, 800)
    
    for i, chunk in enumerate(sub_chunks):
        # Create unique ID using hash
        unique_str = f"{doc_name}_{row['chunk_id']}_{i}_{chunk[:30]}"
        chunk_id = int(hashlib.md5(unique_str.encode()).hexdigest()[:15], 16)
        
        chunks_800.append({
            'id': chunk_id,
            'content': chunk,
            'metadata': json.dumps({
                "document_name": doc_name,
                "source_chunk_id": int(row['chunk_id']),
                "sub_chunk_index": i
            }),
            'chunk_size': 800
        })

print(f"\nChunks created:")
print(f"  200-char chunks: {len(chunks_200)}")
print(f"  800-char chunks: {len(chunks_800)}")

# Insert into tables
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType

schema = StructType([
    StructField("id", LongType(), False),
    StructField("content", StringType(), True),
    StructField("metadata", StringType(), True),
    StructField("chunk_size", IntegerType(), True)
])

# Insert 200-char chunks
df_200 = spark.createDataFrame(chunks_200, schema)
df_200.write.mode("overwrite").saveAsTable("hr_catalog.hr_core.hr_document_chunks_200")
print(f"\n✓ Inserted {len(chunks_200)} chunks into hr_document_chunks_200")

# Insert 800-char chunks
df_800 = spark.createDataFrame(chunks_800, schema)
df_800.write.mode("overwrite").saveAsTable("hr_catalog.hr_core.hr_document_chunks_800")
print(f"✓ Inserted {len(chunks_800)} chunks into hr_document_chunks_800")

In [0]:
%pip install databricks-vectorsearch --quiet

In [0]:
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()

# Sync both indexes
indexes = [
    "hr_catalog.hr_core.hr_rag_index_200",
    "hr_catalog.hr_core.hr_rag_index_800"
]

for idx_name in indexes:
    print(f"Syncing {idx_name}...")
    w.vector_search_indexes.sync_index(index_name=idx_name)
    print(f"  ✓ Sync triggered")

print("\nWaiting for indexes to sync (checking every 10 seconds)...")
for attempt in range(30):  # Max 5 minutes
    time.sleep(10)
    all_ready = True
    print(f"\nCheck #{attempt + 1}:")
    
    for idx_name in indexes:
        index = w.vector_search_indexes.get_index(index_name=idx_name)
        print(f"  {idx_name}:")
        print(f"    Ready: {index.status.ready}")
        print(f"    Indexed rows: {index.status.indexed_row_count}")
        
        if not index.status.ready or index.status.indexed_row_count == 0:
            all_ready = False
    
    if all_ready:
        print("\n✓ All indexes are ready!")
        break
else:
    print("\n⚠ Indexes still syncing after 5 minutes. Continuing anyway...")

In [0]:
from databricks.sdk import WorkspaceClient
import json

# Initialize client
w = WorkspaceClient()

# Test questions
questions = [
    "What are the GDPR requirements for employee data?",
    "What expenses are covered under the travel policy?",
    "What are the required security awareness activities?",
    "How does the performance calibration process work?",
    "What steps must be completed during employee exit?"
]

# Query both indexes
results_comparison = []

for question in questions:
    print(f"\nQuerying: {question}")
    print("="*80)
    
    # Query 200-char index
    result_200 = w.vector_search_indexes.query_index(
        index_name="hr_catalog.hr_core.hr_rag_index_200",
        query_text=question,
        columns=["id", "content", "metadata"],
        num_results=3
    )
    
    # Query 800-char index
    result_800 = w.vector_search_indexes.query_index(
        index_name="hr_catalog.hr_core.hr_rag_index_800",
        query_text=question,
        columns=["id", "content", "metadata"],
        num_results=3
    )
    
    # Extract top results
    top_200 = result_200.result.data_array[0] if result_200.result and result_200.result.data_array else []
    top_800 = result_800.result.data_array[0] if result_800.result and result_800.result.data_array else []
    
    # Get document names and scores
    doc_200 = json.loads(top_200[2]).get('document_name', 'N/A') if len(top_200) > 2 else 'No result'
    doc_800 = json.loads(top_800[2]).get('document_name', 'N/A') if len(top_800) > 2 else 'No result'
    
    score_200 = top_200[3] if len(top_200) > 3 else 0.0
    score_800 = top_800[3] if len(top_800) > 3 else 0.0
    
    # Snippet preview
    snippet_200 = top_200[1][:100] + '...' if len(top_200) > 1 and top_200[1] else 'N/A'
    snippet_800 = top_800[1][:100] + '...' if len(top_800) > 1 and top_800[1] else 'N/A'
    
    # Determine winner
    if score_200 > score_800:
        winner = "200-char"
    elif score_800 > score_200:
        winner = "800-char"
    else:
        winner = "Tie"
    
    results_comparison.append({
        'question': question,
        'index_200_doc': doc_200,
        'index_200_score': round(score_200, 4),
        'index_200_snippet': snippet_200,
        'index_800_doc': doc_800,
        'index_800_score': round(score_800, 4),
        'index_800_snippet': snippet_800,
        'winner': winner,
        'score_diff': round(abs(score_200 - score_800), 4)
    })
    
    print(f"  200-char: {doc_200} (score: {score_200:.4f})")
    print(f"  800-char: {doc_800} (score: {score_800:.4f})")
    print(f"  Winner: {winner}")

print("\n" + "="*80)
print("COMPARISON COMPLETE")
print("="*80)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# Convert results to DataFrame for better visualization
df_comparison = pd.DataFrame(results_comparison)

# Display full comparison table
print("\n" + "="*100)
print("DETAILED COMPARISON TABLE")
print("="*100)
print(f"\n{'Question':<60} {'200-char':<10} {'800-char':<10} {'Winner':<10} {'Δ Score':<10}")
print("-"*100)

for _, row in df_comparison.iterrows():
    q_short = row['question'][:57] + "..." if len(row['question']) > 60 else row['question']
    print(f"{q_short:<60} {row['index_200_score']:<10.4f} {row['index_800_score']:<10.4f} {row['winner']:<10} {row['score_diff']:<10.4f}")

# Summary statistics
print("\n" + "="*100)
print("SUMMARY STATISTICS")
print("="*100)
wins_200 = (df_comparison['winner'] == '200-char').sum()
wins_800 = (df_comparison['winner'] == '800-char').sum()
ties = (df_comparison['winner'] == 'Tie').sum()

avg_score_200 = df_comparison['index_200_score'].mean()
avg_score_800 = df_comparison['index_800_score'].mean()
avg_diff = df_comparison['score_diff'].mean()
max_diff = df_comparison['score_diff'].max()

print(f"\nWins by chunk size:")
print(f"  200-char chunks: {wins_200} questions")
print(f"  800-char chunks: {wins_800} questions")
print(f"  Ties: {ties} questions")

print(f"\nAverage similarity scores:")
print(f"  200-char index: {avg_score_200:.4f}")
print(f"  800-char index: {avg_score_800:.4f}")
print(f"  Average difference: {avg_diff:.4f}")
print(f"  Maximum difference: {max_diff:.4f}")

# Visualize score comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Score comparison by question
ax1 = axes[0]
x = range(len(df_comparison))
width = 0.35
ax1.bar([i - width/2 for i in x], df_comparison['index_200_score'], width, label='200-char', alpha=0.8, color='steelblue')
ax1.bar([i + width/2 for i in x], df_comparison['index_800_score'], width, label='800-char', alpha=0.8, color='coral')
ax1.set_xlabel('Question Number')
ax1.set_ylabel('Cosine Similarity Score')
ax1.set_title('Score Comparison Across Questions')
ax1.set_xticks(x)
ax1.set_xticklabels([f'Q{i+1}' for i in x])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.5, 0.75)

# Plot 2: Score differences
ax2 = axes[1]
colors = ['steelblue' if w == '200-char' else 'coral' if w == '800-char' else 'gray' 
          for w in df_comparison['winner']]
ax2.bar(x, df_comparison['score_diff'], color=colors, alpha=0.8)
ax2.set_xlabel('Question Number')
ax2.set_ylabel('Score Difference (|200 - 800|)')
ax2.set_title('Score Differences by Question')
ax2.set_xticks(x)
ax2.set_xticklabels([f'Q{i+1}' for i in x])
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n{'='*100}")
print("INSIGHTS")
print("="*100)
print(f"\n• The scores are very close overall (avg difference: {avg_diff:.4f})")
print(f"• Both chunk sizes achieve good relevance (0.59-0.69 range)")
print(f"• 200-char chunks slightly outperformed on {wins_200}/5 questions")
if avg_score_200 > avg_score_800:
    print(f"• 200-char chunks have a slight edge in average score (+{avg_score_200 - avg_score_800:.4f})")
else:
    print(f"• 800-char chunks have a slight edge in average score (+{avg_score_800 - avg_score_200:.4f})")
